In [31]:
import pandas as pd
import numpy as np
import time
from itertools import combinations

# =====================================================================
# 1. CORE DISCOVERY MODULES (The "TANE" Black-Box)
# =====================================================================
class Apriori_Gen:
    @staticmethod
    def powerset(s):
        """Generates all subsets of a set [cite: 249-251]."""
        x = len(s)
        return [tuple(s[j] for j in range(x) if (i & (1 << j))) for i in range(1 << x)]

    @classmethod
    def oneUp(cls, C_km1):
        """Generates candidates of size k from size k-1 [cite: 253-256]."""
        if not C_km1: return []
        flat_list = list(set([item for sublist in C_km1 for item in sublist]))
        k = len(C_km1[0]) + 1
        return [list(comb) for comb in combinations(flat_list, k)]

def card_of_partition(candidate, df):
    """Computes partition cardinality [cite: 263-266]."""
    if len(candidate) == 1:
        return df[candidate[0]].nunique()
    return df.drop_duplicates(list(candidate)).shape[0]

def prune_logic(C_k, E, Closure, U):
    """Applies the 4 TANE Pruning Rules [cite: 279-293, 397-440]."""
    sets_to_remove = []

    for S in C_k:
        S_fset = frozenset(S)
        subsets = [frozenset(x) for x in Apriori_Gen.powerset(S) if 0 < len(x) < len(S)]

        for X in subsets:
            # Rule 3: Closure Union (Knowledge Passing) [cite: 416-427]
            Closure[S_fset] = Closure[S_fset].union(Closure[X].difference(X))

            # Rule 1 (Equivalence), Rule 2 (Subset), Rule 4 (Full Key) [cite: 400-415, 428-433]
            is_equiv = any(set(X) == set(equiv[1]) for equiv in E)
            is_subset = set(S).issubset(Closure[X])
            is_key = set(U) == Closure[S_fset]

            if is_equiv or is_subset or is_key:
                sets_to_remove.append(S)
                break # Move to next candidate

    return [c for c in C_k if c not in sets_to_remove], Closure

# =====================================================================
# 2. HYBRID ORCHESTRATION ENGINE (Sampling & Memory Management)
# =====================================================================
def find_fds_tane(df_run, K_limit=3):
    """Executes the level-wise lattice traversal [cite: 104-149]."""
    U = list(df_run.columns)
    k = 1
    C = {1: [[item] for item in U]}
    Closure = {frozenset([item]): {item} for item in U}
    Cardinality = {frozenset([item]): card_of_partition([item], df_run) for item in U}

    F, E = [], []

    while k <= K_limit and k in C and C[k]:
        # Step A: Dependency Validation [cite: 118-149]
        for Cand in C[k]:
            Cand_fset = frozenset(Cand)
            for v_i in set(U).difference(Closure[Cand_fset]):
                comb = Cand + [v_i]
                comb_fset = frozenset(comb)

                # Lazy load cardinality
                if comb_fset not in Cardinality:
                    Cardinality[comb_fset] = card_of_partition(comb, df_run)

                if Cardinality[Cand_fset] == Cardinality[comb_fset]:
                    Closure[Cand_fset].add(v_i)
                    F.append((tuple(Cand), v_i))

        # Note: Step B (Equivalences) is simplified here for brevity

        # Step C & E: Generate next level and Prune [cite: 156-186]
        if k < K_limit:
            C_next = Apriori_Gen.oneUp(C[k])
            # Initialize closures for next level
            for cand in C_next: Closure[frozenset(cand)] = set(cand)
            C[k+1], Closure = prune_logic(C_next, E, Closure, U)

        # Step D: Memory Management (Dereference k-2) [cite: 380-396]
        if k > 1 and (k-2) in C:
            del C[k-2] # Free memory [cite: 387-388]

        k += 1

    return F, E

def analyze_scalable_fds(DF, SS=10, MAX_K=3):
    """Manages randomized sampling and interaction matrix [cite: 206-224]."""
    start_time = time.time()
    colsName = list(DF.columns)
    n = len(colsName)
    Uni_FD = set()

    # Phase 2: Dimensionality Management [cite: 206-213]
    if n > SS:
        runs = (n // SS + 1) * 10
        print(f"High-dimensional data detected. Executing {runs} runs with Sample Size {SS} [cite: 209-212].")
        # Simplified probability sampling for demonstration
        colsRun = [list(np.random.choice(colsName, SS, replace=False)) for _ in range(runs)]
    else:
        runs = 1
        colsRun = [colsName]

    # Phase 3: Iterative Discovery [cite: 214-219]
    for idx, subset_cols in enumerate(colsRun):
        df_subset = DF[subset_cols]
        local_F, local_E = find_fds_tane(df_subset, K_limit=MAX_K)
        Uni_FD.update(local_F) # De-duplicate automatically [cite: 221]

    print(f"Discovery complete in {round(time.time() - start_time, 3)}s")
    print(f"Found {len(Uni_FD)} Universal FDs.")
    return list(Uni_FD)

# =====================================================================
# EXAMPLE EXECUTION
# =====================================================================
# df_synthetic = pd.DataFrame(np.random.randint(0,5,size=(200, 6)), columns=list('ABCDEF'))
# fds = analyze_scalable_fds(df_synthetic, SS=4, MAX_K=2)

In [32]:
from ucimlrepo import fetch_ucirepo

# Fetching 'Breast Cancer Wisconsin' (30 numeric columns)
data = fetch_ucirepo(id=17).data.original
data.columns = [c.replace(' ', '_') for c in data.columns]

# Use a sample if you want it to finish instantly
# engine = FDDiscoveryEngine(data.sample(500), name="BreastCancer")

fds  = analyze_scalable_fds(data, SS=10, MAX_K=2)

High-dimensional data detected. Executing 40 runs with Sample Size 10 [cite: 209-212].
Discovery complete in 5.48s
Found 7981 Universal FDs.


In [33]:
import pandas as pd
import numpy as np
import time
import json
from itertools import combinations
import plotly.graph_objects as go
import plotly.express as px

# =====================================================================
# 1. ADVANCED DISCOVERY ENGINE (TANE + HYBRID SAMPLING)
# =====================================================================

def card_of_partition(candidate, df):
    """Computes the number of unique groups (partition cardinality)."""
    if len(candidate) == 1:
        return df[candidate[0]].nunique()
    return df.drop_duplicates(list(candidate)).shape[0]

def find_fds_tane(df_run, K_limit=2):
    """
    Level-wise lattice traversal.
    K_limit=2 ensures O(n^2) complexity for fast approximation.
    """
    U = list(df_run.columns)
    k = 1
    # Cardinality cache to avoid redundant groupby calls
    Cardinality = {frozenset([item]): card_of_partition([item], df_run) for item in U}
    Closure = {frozenset([item]): {item} for item in U}
    F = []

    # Level 1: Find 1-on-1 FDs
    for lhs in U:
        lhs_fset = frozenset([lhs])
        for rhs in set(U).difference(lhs_fset):
            comb = [lhs, rhs]
            comb_fset = frozenset(comb)
            if comb_fset not in Cardinality:
                Cardinality[comb_fset] = card_of_partition(comb, df_run)

            if Cardinality[lhs_fset] == Cardinality[comb_fset]:
                F.append((tuple([lhs]), rhs))
                Closure[lhs_fset].add(rhs)

    # Level 2: Find 2-on-1 FDs (The Approximation Layer)
    if K_limit >= 2:
        for (a, b) in combinations(U, 2):
            lhs_list = [a, b]
            lhs_fset = frozenset(lhs_list)
            if lhs_fset not in Cardinality:
                Cardinality[lhs_fset] = card_of_partition(lhs_list, df_run)

            for rhs in set(U).difference(lhs_fset):
                comb = lhs_list + [rhs]
                comb_fset = frozenset(comb)
                if comb_fset not in Cardinality:
                    Cardinality[comb_fset] = card_of_partition(comb, df_run)

                if Cardinality[lhs_fset] == Cardinality[comb_fset]:
                    F.append((tuple(lhs_list), rhs))

    return F

# =====================================================================
# 2. VISUALIZATION & REPORTING SUITE
# =====================================================================

def generate_visual_report(df, fds, dataset_name="Analysis"):
    """Generates Sankey and Scatter plots directly in the notebook."""
    cols = list(df.columns)
    dic = {c: i for i, c in enumerate(cols)}

    fds_1 = [f for f in fds if len(f[0]) == 1]
    fds_2 = [f for f in fds if len(f[0]) == 2]

    # --- Sankey Diagram (1-on-1) ---
    if fds_1:
        source = [dic[f[0][0]] for f in fds_1]
        target = [dic[f[1]] for f in fds_1]
        node_colors = [f"rgb({np.random.randint(50,200)},{np.random.randint(50,200)},{np.random.randint(50,200)})" for _ in cols]

        fig_sankey = go.Figure(go.Sankey(
            node=dict(label=cols, pad=25, thickness=25, color=node_colors),
            link=dict(source=source, target=target, value=[1]*len(source), color="rgba(150,150,150,0.4)")
        ))
        fig_sankey.update_layout(title_text=f"{dataset_name}: 1-on-1 FD Flow", width=1000, height=600)
        fig_sankey.show()

    # --- Scatter Plot (2-on-1) ---
    if fds_2:
        c1, c2, dep = [], [], []
        for (lhs, rhs) in fds_2:
            sorted_lhs = sorted(lhs)
            c1.append(sorted_lhs[0]); c2.append(sorted_lhs[1]); dep.append(rhs)

        fig_scatter = px.scatter(x=c1, y=c2, color=dep, title=f"{dataset_name}: 2-on-1 Determinant Matrix")
        fig_scatter.update_xaxes(categoryorder="array", categoryarray=cols)
        fig_scatter.update_yaxes(categoryorder="array", categoryarray=list(reversed(cols)))
        fig_scatter.update_layout(width=1000, height=600)
        fig_scatter.show()

# =====================================================================
# 3. MAIN RUNNER (Incorporating Hybrid Sampling)
# =====================================================================

def run_experiment(df, name="UCI_Dataset", sample_size=10):
    print(f"--- Running Experiment: {name} ---")
    start = time.time()

    # Implementing the Hybrid Sub-sampling Logic from your paper
    if len(df.columns) > sample_size:
        # Probabilistic convergence via randomized subsets
        all_fds = set()
        for _ in range(5): # Iterative convergence runs
            subset = list(np.random.choice(df.columns, sample_size, replace=False))
            all_fds.update(find_fds_tane(df[subset]))
        fds = list(all_fds)
    else:
        fds = find_fds_tane(df)

    duration = round(time.time() - start, 3)
    print(f"Discovery phase finished in {duration}s. Found {len(fds)} FDs.")

    generate_visual_report(df, fds, dataset_name=name)

    # Export for paper citation
    results = {"dataset": name, "runtime": duration, "fd_count": len(fds)}
    with open(f"{name}_summary.json", "w") as f:
        json.dump(results, f)

# Example: Testing with Iris
# from ucimlrepo import fetch_ucirepo
# iris = fetch_ucirepo(id=53).data.original
# run_experiment(iris, name="Iris_Convergence")

In [34]:
run_experiment(data, name="breast_cancer_data")

--- Running Experiment: breast_cancer_data ---
Discovery phase finished in 0.706s. Found 1651 FDs.


In [35]:
# !pip install ucimlrepo

In [36]:
import pandas as pd
import numpy as np
import time
import json
import plotly.graph_objects as go
import plotly.express as px
from itertools import combinations

# =====================================================================
# 1. THE ADVANCED DISCOVERY ENGINE (As per your Research Paper)
# =====================================================================

def find_fds_advanced(df, max_k=2):
    """
    Implements the TANE-style level-wise discovery.
    Specifically captures the 1-on-1 loops you mentioned.
    """
    U = list(df.columns)
    fds_1, fds_2 = [], []

    # Level 1: Discovering 1-on-1 (The Source of your Sankey Loops)
    for lhs in U:
        for rhs in U:
            if lhs == rhs: continue
            # If unique values of LHS match unique values of (LHS, RHS)
            if (df.groupby(lhs)[rhs].nunique() == 1).all():
                fds_1.append(([lhs], rhs))

    # Level 2: Discovering 2-on-1
    if max_k >= 2:
        for (a, b) in combinations(U, 2):
            for rhs in U:
                if rhs in [a, b]: continue
                if (df.groupby([a, b])[rhs].nunique() == 1).all():
                    fds_2.append(([a, b], rhs))
    return fds_1, fds_2

# =====================================================================
# 2. THE VISUALIZATION SUITE (Your exact Plotly logic)
# =====================================================================

def main(df_input):
    start_time = time.time()

    # Logic to handle your 'Complex Loops'
    fds_1, fds_2 = find_fds_advanced(df_input)

    colsName = list(df_input.columns)
    dic = {c: i for i, c in enumerate(colsName)}
    Uni_FD_set1 = [[f[0], f[1]] for f in fds_1]
    Uni_FD_set2 = [[f[0], f[1]] for f in fds_2]

    print(f"Analysis Complete. Time: {round(time.time() - start_time, 4)}s")

    # --- SANKEY DIAGRAM (The 1-on-1 'Complexity' Visual) ---
    if Uni_FD_set1:
        # We use a higher opacity (0.3) so the overlapping loops are visible
        opacity = 0.3
        source = [dic[item[0][0]] for item in Uni_FD_set1]
        target = [dic[item[1]] for item in Uni_FD_set1]

        # Randomized colors per node as per your original script
        colors_node = [list(np.random.choice(range(256), size=3)) for _ in colsName]
        colors_link = [f"rgba({colors_node[s][0]},{colors_node[s][1]},{colors_node[s][2]},{opacity})" for s in source]
        colors_node_str = [f"rgb({c[0]},{c[1]},{c[2]})" for c in colors_node]

        fig_sankey = go.Figure(go.Sankey(
            node=dict(label=colsName, pad=25, thickness=25, color=colors_node_str),
            link=dict(source=source, target=target, value=[1]*len(source), color=colors_link)
        ))
        fig_sankey.update_layout(title_text="Complex 1-on-1 Dependency Web", width=1200, height=800)
        fig_sankey.show()

    # --- SCATTER PLOT (The 2-on-1 'Matrix' Visual) ---
    if Uni_FD_set2:
        c1, c2, dep = [], [], []
        for item in Uni_FD_set2:
            i, j = dic[item[0][0]], dic[item[0][1]]
            # Sorting ensures the matrix is consistent
            if i < j:
                c1.append(item[0][0]); c2.append(item[0][1])
            else:
                c1.append(item[0][1]); c2.append(item[0][0])
            dep.append(item[1])

        fig_scatter = px.scatter(x=c1, y=c2, color=dep, title="2-on-1 Dependency Interactions")
        fig_scatter.update_xaxes(categoryorder="array", categoryarray=colsName)
        fig_scatter.update_yaxes(categoryorder="array", categoryarray=list(reversed(colsName)))
        fig_scatter.show()



In [37]:
main(data)

Analysis Complete. Time: 20.6186s


In [39]:
from ucimlrepo import fetch_ucirepo

# Fetching 'Breast Cancer Wisconsin' (30 numeric columns)
data = fetch_ucirepo(id=186).data.original
data.columns = [c.replace(' ', '_') for c in data.columns]

# Use a sample if you want it to finish instantly
# engine = FDDiscoveryEngine(data.sample(500), name="BreastCancer")

fds  = main(data)

Analysis Complete. Time: 2.2086s


In [40]:
from ucimlrepo import fetch_ucirepo

# Fetching 'Breast Cancer Wisconsin' (30 numeric columns)
data = fetch_ucirepo(id=73).data.original
data.columns = [c.replace(' ', '_') for c in data.columns]

# Use a sample if you want it to finish instantly
# engine = FDDiscoveryEngine(data.sample(500), name="BreastCancer")

fds  = main(data)

Analysis Complete. Time: 16.9139s


In [41]:
from ucimlrepo import fetch_ucirepo

# Fetching 'Breast Cancer Wisconsin' (30 numeric columns)
data = fetch_ucirepo(id=342).data.original
data.columns = [c.replace(' ', '_') for c in data.columns]

# Use a sample if you want it to finish instantly
# engine = FDDiscoveryEngine(data.sample(500), name="BreastCancer")

fds  = main(data)

KeyboardInterrupt: 

In [42]:
def find_fds_advanced(df, max_k=2):
    U = list(df.columns)
    fds_1, fds_2 = [], []

    # LEVEL 1: 1-on-1 (Including Self-Dependencies for the 'Web' look)
    for lhs in U:
        for rhs in U:
            # REMOVED: if lhs == rhs: continue
            # This allows A -> A, which creates the 'loop' ribbons in Sankey
            if (df.groupby(lhs)[rhs].nunique() == 1).all():
                fds_1.append(([lhs], rhs))

    # LEVEL 2: 2-on-1
    if max_k >= 2:
        for (a, b) in combinations(U, 2):
            for rhs in U:
                # We still exclude RHS being part of LHS for 2-on-1 to avoid clutter
                if rhs in [a, b]: continue
                if (df.groupby([a, b])[rhs].nunique() == 1).all():
                    fds_2.append(([a, b], rhs))
    return fds_1, fds_2

def main(df_input):
    fds_1, fds_2 = find_fds_advanced(df_input)
    colsName = list(df_input.columns)
    dic = {c: i for i, c in enumerate(colsName)}

    # --- SANKEY DIAGRAM WITH LOOPS ---
    if fds_1:
        # High opacity is key for seeing the 'web' overlap
        opacity = 0.4
        source = [dic[f[0][0]] for f in fds_1]
        target = [dic[f[1]] for f in fds_1]

        # We need distinct colors to see the paths clearly
        colors_node = [list(np.random.choice(range(256), size=3)) for _ in colsName]
        colors_link = [f"rgba({colors_node[s][0]},{colors_node[s][1]},{colors_node[s][2]},{opacity})" for s in source]
        colors_node_str = [f"rgb({c[0]},{c[1]},{c[2]})" for c in colors_node]

        fig_sankey = go.Figure(go.Sankey(
            node=dict(label=colsName, pad=15, thickness=20, color=colors_node_str),
            link=dict(source=source, target=target, value=[1]*len(source), color=colors_link)
        ))

        # Use a large width to spread out the 'web'
        fig_sankey.update_layout(
            title_text="Complex Functional Dependency Web (Including Trivial & Cyclic Flows)",
            width=1400, height=900
        )
        fig_sankey.show()

In [43]:
from ucimlrepo import fetch_ucirepo

# Fetching 'Breast Cancer Wisconsin' (30 numeric columns)
data = fetch_ucirepo(id=73).data.original
data.columns = [c.replace(' ', '_') for c in data.columns]

# Use a sample if you want it to finish instantly
# engine = FDDiscoveryEngine(data.sample(500), name="BreastCancer")

fds  = main(data)

In [45]:
from ucimlrepo import fetch_ucirepo

# Fetching 'Breast Cancer Wisconsin' (30 numeric columns)
data = fetch_ucirepo(id=186).data.original
data.columns = [c.replace(' ', '_') for c in data.columns]

# Use a sample if you want it to finish instantly
# engine = FDDiscoveryEngine(data.sample(500), name="BreastCancer")

main(data)